In [0]:
from pyspark.sql import functions as F

# Drop Duplicates within water mark period

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS pyspark_catalog.moon.orders_stream_source (
    order_id     STRING,
    customer_id  STRING,
    amount       DOUBLE,
    event_time   TIMESTAMP,
    batch_tag    STRING
) USING DELTA
""")

DataFrame[]

In [0]:
df=spark.readStream.format('delta').table('pyspark_catalog.moon.orders_stream_source')

In [0]:
df=df.withWatermark("event_time","5 minutes").dropDuplicatesWithinWatermark(['order_id'])

In [0]:
df.writeStream\
    .format('delta')\
    .outputMode('append')\
    .option('checkpointLocation',"/Volumes/pyspark_catalog/moon/files/check/")\
    .trigger(once=True)\
    .toTable('pyspark_catalog.moon.orders')

### batch 1

In [0]:
spark.sql("""
INSERT INTO pyspark_catalog.moon.orders_stream_source VALUES
('ORD001', 'C1', 250.0, '2026-08-06T10:00:00', 'batch1'),
('ORD002', 'C2', 400.0, '2026-08-06T10:01:00', 'batch1'),
('ORD003', 'C3', 150.0, '2026-08-06T10:02:00', 'batch1')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

### batch 2

In [0]:
spark.sql("""
INSERT INTO pyspark_catalog.moon.orders_stream_source VALUES
('ORD001', 'C1', 250.0, '2026-08-06T10:00:00', 'batch2_dupe'),
('ORD004', 'C4', 300.0, '2026-08-06T10:03:00', 'batch2_new')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

### batch 3

In [0]:
spark.sql("""
INSERT INTO pyspark_catalog.moon.orders_stream_source VALUES
('ORD001', 'C1', 250.0, '2026-08-06T10:00:00', 'batch3_late_dupe'),
('ORD005', 'C5', 500.0, '2026-08-06T11:00:00', 'batch3_new')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from pyspark_catalog.moon.orders

order_id,customer_id,amount,event_time,batch_tag
ORD005,C5,500.0,2026-08-06T11:00:00.000Z,batch3_new
ORD004,C4,300.0,2026-08-06T10:03:00.000Z,batch2_new
ORD001,C1,250.0,2026-08-06T10:00:00.000Z,batch1
ORD002,C2,400.0,2026-08-06T10:01:00.000Z,batch1
ORD003,C3,150.0,2026-08-06T10:02:00.000Z,batch1
